# Local PDF RAG System

A simple Retrieval-Augmented Generation (RAG) system that:
- Loads local PDF files
- Creates embeddings using HuggingFace (free & local)
- Stores vectors in FAISS
- Connects to LM Studio for LLM inference

## 1. Install Dependencies

In [19]:
# Install required packages
!pip install langchain langchain-community langchain-openai langchain-text-splitters langchain-huggingface langchain-core pypdf faiss-cpu sentence-transformers -q
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q

ERROR: Could not find a version that satisfies the requirement torchaudio (from versions: none)
ERROR: No matching distribution found for torchaudio


## 2. Import Libraries

In [20]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import torch
import os

# Check GPU availability
def get_device():
    """Detect and return the best available device."""
    if torch.cuda.is_available():
        device = "cuda"
        gpu_name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"GPU Detected: {gpu_name} ({vram:.1f} GB VRAM)")
    else:
        device = "cpu"
        print("No GPU detected, using CPU")
    return device

DEVICE = get_device()

No GPU detected, using CPU


## 3. Configuration

In [21]:
# Chunking settings
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100

# Embedding model (runs locally, free)
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# LM Studio connection settings
LM_STUDIO_BASE_URL = "http://localhost:1234/v1"
LM_STUDIO_API_KEY = "not-needed"

# GPU Settings
EMBEDDING_BATCH_SIZE = 64  # Increase for faster processing with GPU

## 4. Load and Process PDF

In [22]:
def load_pdf(pdf_path: str):
    """Load PDF document using PyPDFLoader."""
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF file not found: {pdf_path}")
    
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    print(f"Loaded {len(documents)} pages from PDF")
    return documents

In [23]:
def split_documents(documents, chunk_size: int = CHUNK_SIZE, chunk_overlap: int = CHUNK_OVERLAP):
    """Split documents into chunks using RecursiveCharacterTextSplitter."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = text_splitter.split_documents(documents)
    print(f"Split into {len(chunks)} chunks")
    return chunks

## 5. Create Embeddings and Vector Store

In [24]:
def create_embeddings():
    """Initialize HuggingFace embeddings model with GPU acceleration."""
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={
            'device': DEVICE,
            'trust_remote_code': True
        },
        encode_kwargs={
            'normalize_embeddings': True,
            'batch_size': EMBEDDING_BATCH_SIZE
        }
    )
    print(f"Initialized embedding model: {EMBEDDING_MODEL} on {DEVICE.upper()}")
    return embeddings

In [25]:
def create_vector_store(chunks, embeddings):
    """Create FAISS vector store from document chunks."""
    vector_store = FAISS.from_documents(chunks, embeddings)
    print("Created FAISS vector store")
    return vector_store

## 6. Connect to LM Studio LLM

In [26]:
def create_llm():
    """Create ChatOpenAI instance connected to LM Studio."""
    llm = ChatOpenAI(
        base_url=LM_STUDIO_BASE_URL,
        api_key=LM_STUDIO_API_KEY,
        temperature=0.7
    )
    print("Connected to LM Studio")
    return llm

## 7. Create RAG Chain

In [27]:
# Prompt template that forces the LLM to answer ONLY based on context
PROMPT_TEMPLATE = """Use the following pieces of context to answer the question at the end.
If you don't know the answer based on the context, just say that you don't know.
Do NOT try to make up an answer. Only use information from the provided context.

Context:
{context}

Question: {question}

Answer based ONLY on the context above:"""

PROMPT = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

In [28]:
def format_docs(docs):
    """Format retrieved documents into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)

def create_rag_chain(vector_store, llm):
    """Create RAG chain using LCEL (LangChain Expression Language)."""
    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 4}  # Return top 4 relevant chunks
    )
    
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | PROMPT
        | llm
        | StrOutputParser()
    )
    print("Created RAG chain")
    return rag_chain, retriever

## 8. Main RAG System Class

In [29]:
class LocalPDFRAG:
    """Local PDF RAG System using LangChain, FAISS, and LM Studio."""
    
    def __init__(self, pdf_path: str):
        """Initialize the RAG system with a PDF file."""
        print("Initializing Local PDF RAG System...")
        print("-" * 50)
        
        # Load and process PDF
        documents = load_pdf(pdf_path)
        chunks = split_documents(documents)
        
        # Create embeddings and vector store
        self.embeddings = create_embeddings()
        self.vector_store = create_vector_store(chunks, self.embeddings)
        
        # Connect to LLM and create RAG chain
        self.llm = create_llm()
        self.rag_chain, self.retriever = create_rag_chain(self.vector_store, self.llm)
        
        print("-" * 50)
        print("RAG System Ready!")
    
    def ask(self, question: str) -> str:
        """Ask a question and get an answer based on the PDF content."""
        return self.rag_chain.invoke(question)
    
    def ask_with_sources(self, question: str) -> dict:
        """Ask a question and get answer with source documents."""
        answer = self.rag_chain.invoke(question)
        docs = self.retriever.invoke(question)
        return {
            "answer": answer,
            "sources": [
                {
                    "page": doc.metadata.get("page", "N/A"),
                    "content": doc.page_content[:200] + "..."
                }
                for doc in docs
            ]
        }
    
    def similarity_search(self, query: str, k: int = 4) -> list:
        """Perform similarity search and return relevant chunks."""
        docs = self.vector_store.similarity_search(query, k=k)
        return [
            {
                "page": doc.metadata.get("page", "N/A"),
                "content": doc.page_content
            }
            for doc in docs
        ]
    
    def save_vector_store(self, path: str = "faiss_index"):
        """Save the FAISS index to disk."""
        self.vector_store.save_local(path)
        print(f"Vector store saved to {path}")
    
    @classmethod
    def load_from_index(cls, index_path: str):
        """Load RAG system from a saved FAISS index."""
        embeddings = create_embeddings()
        vector_store = FAISS.load_local(
            index_path,
            embeddings,
            allow_dangerous_deserialization=True
        )
        llm = create_llm()
        
        instance = cls.__new__(cls)
        instance.embeddings = embeddings
        instance.vector_store = vector_store
        instance.llm = llm
        instance.rag_chain, instance.retriever = create_rag_chain(vector_store, llm)
        
        print("Loaded RAG system from saved index")
        return instance

## 9. Usage Example

In [30]:
# Ask for PDF file path
def get_pdf_path():
    """Prompt user for PDF file path and validate it exists."""
    while True:
        pdf_path = input("Enter the path to your PDF file: ").strip()
        
        # Remove quotes if user wrapped the path in them
        pdf_path = pdf_path.strip('"').strip("'")
        
        # Handle Windows paths - normalize slashes and fix common issues
        pdf_path = pdf_path.replace('/', '\\')  # Normalize to backslashes
        pdf_path = os.path.normpath(pdf_path)   # Clean up the path
        
        # Handle paths copied with "Copy as path" (often have extra quotes)
        if pdf_path.startswith('"') or pdf_path.startswith("'"):
            pdf_path = pdf_path[1:]
        if pdf_path.endswith('"') or pdf_path.endswith("'"):
            pdf_path = pdf_path[:-1]
        
        if not pdf_path:
            print("Please enter a valid path.")
            continue
        
        if not os.path.exists(pdf_path):
            print(f"File not found: {pdf_path}")
            print("Tip: Right-click the PDF in Explorer → 'Copy as path', then paste here.")
            continue
        
        if not pdf_path.lower().endswith('.pdf'):
            response = input("Warning: File does not have .pdf extension. Continue anyway? (y/n): ").strip()
            if response.lower() != 'y':
                continue
        
        print(f"Found: {pdf_path}")
        return pdf_path

# Get PDF path from user and initialize the RAG system
pdf_path = get_pdf_path()
rag = LocalPDFRAG(pdf_path)

Found: C:\Users\SAUGAT\Downloads\Modern Systems Analysis And Design - Joseph Valacich, Joey George.pdf
Initializing Local PDF RAG System...
--------------------------------------------------
Loaded 545 pages from PDF
Split into 2166 chunks


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8638.02it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Initialized embedding model: sentence-transformers/all-MiniLM-L6-v2 on CPU
Created FAISS vector store
Connected to LM Studio
Created RAG chain
--------------------------------------------------
RAG System Ready!


In [ ]:
# Ask a question interactively
import textwrap

def format_answer(text, width=80):
    """Wrap text for better readability."""
    paragraphs = text.split('\n')
    wrapped = []
    for p in paragraphs:
        if p.strip():
            wrapped.append(textwrap.fill(p, width=width))
        else:
            wrapped.append('')
    return '\n'.join(wrapped)

question = input("❓ Enter your question: ").strip()
if question:
    print("\n🔍 Searching document...")
    answer = rag.ask(question)
    
    print("\n" + "=" * 60)
    print("💬 ANSWER:")
    print("=" * 60)
    print(format_answer(answer))
    print("=" * 60)
else:
    print("⚠️  No question entered.")

In [ ]:
# Ask with sources to see which parts of the PDF were used
import textwrap

def format_text(text, width=80):
    paragraphs = text.split('\n')
    wrapped = [textwrap.fill(p, width=width) if p.strip() else '' for p in paragraphs]
    return '\n'.join(wrapped)

question = input("❓ Enter your question: ").strip()
if question:
    print("\n🔍 Searching document...")
    result = rag.ask_with_sources(question)
    
    print("\n" + "=" * 60)
    print("💬 ANSWER:")
    print("=" * 60)
    print(format_text(result["answer"]))
    
    print("\n" + "-" * 60)
    print("📚 SOURCES:")
    print("-" * 60)
    for i, source in enumerate(result["sources"], 1):
        print(f"\n[{i}] Page {source['page']}:")
        print(format_text(source['content'], width=70))
    print("=" * 60)
else:
    print("⚠️  No question entered.")

In [33]:
# Perform just similarity search without LLM
relevant_chunks = rag.similarity_search("your search query here", k=3)
for chunk in relevant_chunks:
    print(f"Page {chunk['page']}:")
    print(chunk['content'])
    print("-" * 50)

Page 354:
INVENTORY ITEM(Item_Number,Item_Description,Quantity_in_Stock, 
Minimum_Order_Quantity,Type_of_Item)
ITEM SALE(Receipt_Number,Product_ID,Quantity_Sold)
INVOICE ITEM(Vendor_ID,Invoice_Number,Item_Number,Quantity_Added)
RECIPE(Product_ID,Item_Number,Quantity_Used)
VENDOR(Vendor_ID,Vendor_Name)
Vendor
ID Name T ype of Item Total Quantity Added
V1 V1name aaa nnn1
bbb nnn2
ccc nnn3
V2 V2name bbb nnn4
mmm nnn5
x
x
x
Monthly Vendor Load Report Page x of n
for Month: xxxxx
Figure 9-15 
Hoosier Burger Monthly Vendor Load 
Report
--------------------------------------------------
Page 198:
•	 Customer Comments
❑ ❑Company Info
❑ ❑Feedback
❑ ❑Contact Information
❑ ❑User Profile Manager
❑ ❑Order Maintenance Manager
❑ ❑Content (catalog) Manager
❑ ❑Reports
•	 Total Hits
•	 Most Frequent Page Views
•	 Users/Time of Day
•	 Users/Day of Week
•	 Shoppers Not Purchasing (used shopping cart—did 
not checkout)
•	 Feedback Analysis
--------------------------------------------------
Page 362:
locate

In [34]:
# Optional: Save the vector store for later use
rag.save_vector_store("my_pdf_index")

Vector store saved to my_pdf_index


In [35]:
# Optional: Load from saved index (faster than re-processing PDF)
# rag_loaded = LocalPDFRAG.load_from_index("my_pdf_index")

## 10. Interactive Q&A Loop

In [ ]:
import textwrap
from IPython.display import display, Markdown, clear_output

def format_answer(text, width=80):
    """Wrap text for better readability."""
    paragraphs = text.split('\n')
    wrapped = []
    for p in paragraphs:
        if p.strip():
            wrapped.append(textwrap.fill(p, width=width))
        else:
            wrapped.append('')
    return '\n'.join(wrapped)

def interactive_qa(rag_system):
    """Run an interactive Q&A session with better formatting."""
    print("=" * 60)
    print("        📄 PDF Q&A Session")
    print("=" * 60)
    print("Type your question and press Enter.")
    print("Type 'quit', 'exit', or 'q' to end the session.")
    print("=" * 60)
    
    while True:
        print()
        question = input("❓ Your question: ").strip()
        
        if question.lower() in ['quit', 'exit', 'q']:
            print("\n👋 Ending session. Goodbye!")
            break
        
        if not question:
            print("⚠️  Please enter a valid question.")
            continue
        
        print("\n🔍 Searching document...")
        answer = rag_system.ask(question)
        
        print("\n" + "-" * 60)
        print("💬 ANSWER:")
        print("-" * 60)
        print(format_answer(answer))
        print("-" * 60)

# Run interactive session
interactive_qa(rag)